# Holdout Testing

Evaluates APULA's ability to reproduce experimental outcomes using investigation-level and individual observations held out from the evaluation process (**out-of-sample validation**).

*Ambiguous labels are excluded. Different experimental datasets are evaluated separately to avoid misleading comparisons.*

### Imports

In [1]:
import subprocess
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

ROOT = Path.cwd()

NODE_DIR = r"C:\Program Files\nodejs" # This is just here cause of where nodejs is on my device alright -kira
NPX = os.path.join(NODE_DIR, "npx.cmd")

os.environ["PATH"] = NODE_DIR + os.pathsep + os.environ["PATH"]

def run_typescript(mode, input_data=None):
    p = subprocess.run(
        [NPX, "tsx", "import.ts", mode],
        input=input_data,
        capture_output=True,
        text=True,
        cwd=ROOT
    )

    if p.returncode != 0:
        raise RuntimeError(p.stderr)

    return json.loads(p.stdout)

runs = run_typescript("export")
df = pd.DataFrame(runs)

print(f"Imported {len(df)} runs.")
print(f"DataFrame shape: {df.shape}")

Imported 654 runs.
DataFrame shape: (654, 38)


## Investigation-level holdout
Evaluates model concordance when BASS, BASS-II, and SAFFIRE-3 are left out *(grouped out-of-sample empirical validation)*

In [2]:
from sklearn.metrics import confusion_matrix, classification_report

# Solid-fuel investigations only
groups = ["BASS", "BASS-II", "SAFFIRE-3"]
df = df[df["experimentId"].isin(groups)]

all_predictions = []
fold_metrics = []

for held_out in sorted(df["experimentId"].unique()):
    
    train = df[df["experimentId"] != held_out]
    test = df[df["experimentId"] == held_out]

    predictions = run_typescript(
        "predict",
        test.to_json(orient="records")
    )

    results = pd.DataFrame(predictions)
    results["held_out"] = held_out

    # Scores only observations with known experimental outcomes
    valid = results["observedIgnition"].notna()

    if valid.any():
        accuracy = (
            results.loc[valid, "observedIgnition"].astype(bool)
            ==
            results.loc[valid, "predictedIgnition"].astype(bool)
        ).mean() * 100
    else:
        accuracy = None

    fold_metrics.append({
        "held_out": held_out,
        "test_rows": len(test),
        "scored_rows": int(valid.sum()),
        "accuracy_pct": accuracy
    })

    all_predictions.append(results)

### Results

In [3]:

validation_results = pd.concat(all_predictions, ignore_index=True)

summary_rows = []

for group, data in validation_results.groupby("held_out"):
    y_true = data["observedIgnition"].astype(bool)
    y_pred = data["predictedIgnition"].astype(bool)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    summary_rows.append({
        "Held-out Group": group,
        "Observations": len(data),
        "Observed Positive": y_true.sum(),
        "Observed Negative": (~y_true).sum(),
        "True Positives": tp,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "Accuracy": (y_true == y_pred).mean()
    })

summary_table = pd.DataFrame(summary_rows)

summary_table["Accuracy"] = (
    summary_table["Accuracy"].map(lambda x: f"{x:.2%}")
)

print("===== VALIDATION SUMMARY =====")
display(summary_table)

print("\n===== CLASSIFICATION REPORTS =====")

for group, data in validation_results.groupby("held_out"):
    y_true = data["observedIgnition"].astype(bool)
    y_pred = data["predictedIgnition"].astype(bool)

    report = classification_report(
        y_true,
        y_pred,
        labels=[False, True],
        target_names=["No ignition", "Ignition"],
        zero_division=0,
        output_dict=True
    )

    report_table = pd.DataFrame(report).T.round(3)

    print(f"\n===== {group} =====")
    display(report_table)

===== VALIDATION SUMMARY =====


,Held-out Group,Observations,Observed Positive,Observed Negative,True Positives,True Negatives,False Positives,False Negatives,Accuracy
0,BASS,122,122,0,116,0,0,6,95.08%
1,BASS-II,129,127,2,124,1,1,3,96.90%
2,SAFFIRE-3,2,2,0,2,0,0,0,100.00%



===== CLASSIFICATION REPORTS =====

===== BASS =====


,precision,recall,f1-score,support
No ignition,0.000,0.000,0.000,0.000
Ignition,1.000,0.951,0.975,122.000
accuracy,0.951,0.951,0.951,0.951
macro avg,0.500,0.475,0.487,122.000
weighted avg,1.000,0.951,0.975,122.000



===== BASS-II =====


,precision,recall,f1-score,support
No ignition,0.250,0.500,0.333,2.000
Ignition,0.992,0.976,0.984,127.000
accuracy,0.969,0.969,0.969,0.969
macro avg,0.621,0.738,0.659,129.000
weighted avg,0.980,0.969,0.974,129.000



===== SAFFIRE-3 =====


,precision,recall,f1-score,support
No ignition,0.0,0.0,0.0,0.0
Ignition,1.0,1.0,1.0,2.0
accuracy,1.0,1.0,1.0,1.0
macro avg,0.5,0.5,0.5,2.0
weighted avg,1.0,1.0,1.0,2.0


In [4]:
vr = pd.concat(all_predictions, ignore_index=True)

fn = vr[
    vr["observedIgnition"].astype(bool) &
    ~vr["predictedIgnition"].astype(bool)
].copy()

print("False negatives:", len(fn))
display(fn.groupby("held_out").size().to_frame("count"))

bass2 = vr[vr["held_out"] == "BASS-II"].copy()

bass2["correct"] = (
    bass2["observedIgnition"].astype(bool) ==
    bass2["predictedIgnition"].astype(bool)
)

print("BASS-II: correct vs incorrect")

display(
    bass2.groupby("correct")[
        ["oxygenMarginPct", "predictedFlameSpreadMmS"]
    ].agg(["count", "mean", "median", "min", "max"]).round(3)
)

display(
    bass2.groupby("correct")["oxygenMarginPct"]
    .describe()
    .round(3)
)

False negatives: 9


,count
held_out,
BASS,6
BASS-II,3


BASS-II: correct vs incorrect


oxygenMarginPct                           predictedFlameSpreadMmS  \
                  count   mean median   min   max                   count   
correct                                                                     
False                 4  0.190  -0.22 -0.84  2.04                       4   
True                125  3.802   3.43 -1.05  7.96                     125   

                                   
          mean median  min    max  
correct                            
False    0.915    0.0  0.0   3.66  
True     5.549    4.8  0.0  25.50

,count,mean,std,min,25%,50%,75%,max
correct,,,,,,,,
False,4.0,0.190,1.268,-0.84,-0.412,-0.22,0.382,2.04
True,125.0,3.802,2.171,-1.05,2.300,3.43,5.860,7.96


## Individual-level hold out
Evaluates model concordance on an individual level *(individual-level out-of-sample empirical validation)*  

*//also cus the categorical level ones are suspishaaas*

In [5]:
from sklearn.model_selection import train_test_split

# Preserve the original LOIO results
individual_df = df.copy()

# Split individual observations within each investigation
train_parts = []
test_parts = []

for group in groups:
    group_data = individual_df[
        individual_df["experimentId"] == group
    ]

    train_group, test_group = train_test_split(
        group_data,
        test_size=0.20, # 80:20 train-test split
        random_state=0
    )

    train_parts.append(train_group)
    test_parts.append(test_group)

individual_train = pd.concat(train_parts).reset_index(drop=True)
individual_test = pd.concat(test_parts).reset_index(drop=True)

print("Training observations:", len(individual_train))
print("Held-out observations:", len(individual_test))

print("\nTraining distribution:")
display(individual_train["experimentId"].value_counts())

print("\nHoldout distribution:")
display(individual_test["experimentId"].value_counts())

Training observations: 201
Held-out observations: 52

Training distribution:


experimentId
BASS-II      103
BASS          97
SAFFIRE-3      1
Name: count, dtype: int64


Holdout distribution:


experimentId
BASS-II      26
BASS         25
SAFFIRE-3     1
Name: count, dtype: int64

### Results

In [6]:
individual_predictions = run_typescript(
    "predict",
    individual_test.to_json(orient="records")
)

individual_results = pd.DataFrame(individual_predictions)

print("Predictions generated:", len(individual_results))
display(individual_results.head())

Predictions generated: 52


,experimentId,testNumber,observedIgnition,sourceIgnitionStatus,predictedIgnition,oxygenMarginPct,predictedFlameSpreadMmS,observedFlameSpreadMmS
0,BASS,115-r85,True,Ignited,True,6.9,12.4,None
1,BASS,105-r98,True,Ignited,True,6.9,12.4,None
2,BASS,21-r11,True,Ignited,True,6.9,12.4,None
3,BASS,83c-r72,True,Ignited,True,6.9,12.4,None
4,BASS,19-r9,True,Ignited,True,6.9,12.4,None


In [7]:
from sklearn.metrics import confusion_matrix, classification_report

valid = individual_results["observedIgnition"].notna()

scored = individual_results.loc[valid].copy()

y_true = scored["observedIgnition"].astype(bool)
y_pred = scored["predictedIgnition"].astype(bool)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[False, True]
).ravel()

print("===== INDIVIDUAL RUN HOLDOUT =====")
print("Scored observations:", len(scored))
print("True positives:", tp)
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print(f"Accuracy: {(y_true == y_pred).mean():.2%}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[False, True],
    target_names=["No ignition", "Ignition"],
    zero_division=0
))

cols = [
    "experimentId", "testNumber",
    "rawFuelMaterial", "baseMaterial",
    "geometryType", "thicknessMm",
    "initialO2Pct", "pressurePsi",
    "gravityG", "initialAirflowCmS",
    "finalAirflowCmS",
    "observedIgnition", "predictedIgnition"
]

===== INDIVIDUAL RUN HOLDOUT =====
Scored observations: 52
True positives: 50
True negatives: 1
False positives: 0
False negatives: 1
Accuracy: 98.08%

Classification report:
              precision    recall  f1-score   support

 No ignition       0.50      1.00      0.67         1
    Ignition       1.00      0.98      0.99        51

    accuracy                           0.98        52
   macro avg       0.75      0.99      0.83        52
weighted avg       0.99      0.98      0.98        52

